In [1]:
from leorbit.api import get_satellite, get_passes, Timestamp, Quantity, GPS, VisibleFromEarthLocationEvent, TimeInterval

Let's compute the position of a LEO satellite! Let's choose the ISS for the demonstration. 
Its NORAD Catalog ID is 25544.

If you have no idea of what a NORAD catalog ID is, you should definitely take 5min to check [Celestrak.org](https://celestrak.org)

In [11]:
iss = get_satellite(25544)
now = Timestamp.now()
c = iss.coordinates(now)

c.gps()

<GPS:  088° 02′ 02″E,  050° 22′ 57″N>

Now that we have computed its position for the given time, we can project
it into any frame we want!

In [12]:
c.gcrf().human_repr("km")

'<Vector3 x=-2037.4902123747336 y=-3818.932044072138 z=5228.989900723145 [km]>'

In [13]:
c.itrf().human_repr("km")

'<Vector3 x=148.48690303361718 y=4325.917239429267 z=5228.989900723145 [km]>'

Horizontal coordinates in any Earth local frame!
For example, let's try in Paris.

In [15]:
gps_paris = GPS(
    longitude=2.333333 * Quantity.degree, 
    latitude=48.866667 * Quantity.degree, 
    altitude=0 * Quantity.meter
)

c.horizontal(gps_paris.earth_local_frame)

<Horizontal: Azimuth:  053° 29′ 16″, Altitude: - 022° 24′ 24″>

Ugh, negative altitude, it means it is not visible currently...

Well, I'd like to know next time it is visible in my area. Let's check when that happens, the next 7 days!

In [16]:
timeline = TimeInterval(
    start=now,
    stop=now + 7 * Quantity.day,
    dt=5 * Quantity.second
)
first_pass, *others = get_passes(iss, timeline, gps_paris, altitude_angle_min_degrees=0.)
first_pass

<TimeInterval from: '2026-05-08 at 20:26:37' to: '2026-05-08 at 20:36:42' dt: 5s>

Great! Now let's export the trajectory for the next pass:

In [17]:
iss.trajectory(first_pass).horizontal(gps_paris.earth_local_frame).to_csv(first_pass)

'timestamp,azimuth,elevation,range\n2026-05-08T20:26:37.493750+00:00,-147.8,0.2,6788.3\n2026-05-08T20:26:42.493750+00:00,-148.1,0.5,6788.3\n2026-05-08T20:26:47.493750+00:00,-148.4,0.8,6788.2\n2026-05-08T20:26:52.493750+00:00,-148.8,1.1,6788.2\n2026-05-08T20:26:57.493750+00:00,-149.1,1.4,6788.2\n2026-05-08T20:27:02.493750+00:00,-149.5,1.7,6788.1\n2026-05-08T20:27:07.493750+00:00,-149.8,2.0,6788.1\n2026-05-08T20:27:12.493750+00:00,-150.2,2.4,6788.1\n2026-05-08T20:27:17.493750+00:00,-150.6,2.7,6788.0\n2026-05-08T20:27:22.493750+00:00,-151.0,3.0,6788.0\n2026-05-08T20:27:27.493750+00:00,-151.4,3.4,6788.0\n2026-05-08T20:27:32.493750+00:00,-151.8,3.7,6787.9\n2026-05-08T20:27:37.493750+00:00,-152.3,4.1,6787.9\n2026-05-08T20:27:42.493750+00:00,-152.8,4.4,6787.9\n2026-05-08T20:27:47.493750+00:00,-153.2,4.8,6787.8\n2026-05-08T20:27:52.493750+00:00,-153.7,5.1,6787.8\n2026-05-08T20:27:57.493750+00:00,-154.2,5.5,6787.8\n2026-05-08T20:28:02.493750+00:00,-154.8,5.9,6787.8\n2026-05-08T20:28:07.493750+0